In [1]:
#============== Install required libraries==============#
# !pip install --upgrade pip 
# !pip install opencv-python matplotlib

# Import standard libraries #
import os                                             # Work with file paths and directories
import cv2                                            # OpenCV for image/video loading, processing, tracking
import torch                                          # PyTorch for deep learning and tensor computations
import numpy as np                                    # numerical operations and arrays
import matplotlib.pyplot as plt                       #plotting graphs & visualizing images
from scipy.signal import savgol_filter                #library applies a Savitzky-Golay filter to a signal

In [ ]:
#======================= Settings ==================#
# Original experiment notebook (kept for reference). See demo.ipynb for the package-based version.
# Input video: put your video in the "examples/" folder and set its file name here.
VIDEO_NAME = "bicycle.mp4"
video_path = os.path.join("..", "examples", VIDEO_NAME)   # run from notebooks/

# Output file for the tracked video
OUTPUT_PATH = os.path.join("..", "results", "tracking_final.mp4")

# Frames whose longer side is bigger than this are downscaled before tracking,
# so long / high-resolution videos fit in GPU (or CPU) memory. Set to None to disable.
MAX_SIDE = 640

# Alternative (Google Colab upload) #
# from google.colab import files
# uploaded = files.upload()              # Opens a dialog to upload a file from your computer
# video_path = list(uploaded.keys())[0]  # Gets the name of the first uploaded file
# print("✅ Video uploaded:", video_path) # Confirms which video was uploaded

In [ ]:
# =====================Load Frames=========================== #
cap = cv2.VideoCapture(video_path)  # Open the video file
frames = []                         # Empty list to store all video frames

# Check if video file opened successfully
if not cap.isOpened():
    raise FileNotFoundError(f"❌ Could not open video: {video_path}")

# Keep the source frame rate so the output video plays at the same speed
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0 or np.isnan(fps):
    fps = 30.0                            # fallback if the container has no FPS info

# Loop through video frame by frame
while True:
    ret, frame = cap.read()               # Read one frame from the video
    if not ret:                           # If no more frames to read, break the loop
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # Convert from BGR (OpenCV default) to RGB
    frames.append(frame_rgb)              # Save the frame in the list

cap.release()                             # Release the video file (important to free memory)

if not frames:
    raise ValueError(f"❌ No frames could be read from: {video_path}")

# Downscale large videos (scale = 1.0 means the original size is kept)
H0, W0 = frames[0].shape[:2]
scale = 1.0
if MAX_SIDE is not None and max(H0, W0) > MAX_SIDE:
    scale = MAX_SIDE / max(H0, W0)
    # (W, H) for OpenCV, rounded to even numbers because most video encoders require that
    new_size = (2 * int(round(W0 * scale / 2)), 2 * int(round(H0 * scale / 2)))
    frames = [cv2.resize(f, new_size, interpolation=cv2.INTER_AREA) for f in frames]

# Convert list of frames into a single NumPy array
frames = np.array(frames, dtype=np.uint8)
print("✅ Frames loaded:", frames.shape)  # Show dimensions # (T, H, W, 3)
print(f"   FPS: {fps:.2f} | original size: {W0}x{H0} | scale: {scale:.3f}")

In [4]:
# ===================== Convert to Tensor =======================#
# Check if GPU ("cuda") is available; otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Convert frames (NumPy array) ---> PyTorch tensor
# Shape steps:
#   frames: (T, H, W, 3)
#   permute(0, 3, 1, 2): (T, 3, H, W)  [move color channel before height/width]
#   unsqueeze(0): (1, T, 3, H, W)      [add batch dimension]
#   float(): convert to float32
#   to(device): move to GPU (if available)
video = torch.from_numpy(frames).permute(0, 3, 1, 2).unsqueeze(0).float().to(device)

print("✅ Video tensor shape:", video.shape)  #  Should be (1, T, 3, H, W)

# ==================== Load CoTracker ==========================#
# Load the CoTracker model from PyTorch Hub.
# "online" version: processes frame-by-frame in real time (slower, but streams)
# "offline" version: processes entire video at once (faster, good for Colab)
#cotracker = torch.hub.load("facebookresearch/co-tracker", "cotracker3_online").to(device)
cotracker = torch.hub.load("facebookresearch/co-tracker", "cotracker3_offline").to(device)

✅ Video tensor shape: torch.Size([1, 344, 3, 240, 426])
Downloading: "https://github.com/facebookresearch/co-tracker/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth" to /root/.cache/torch/hub/checkpoints/scaled_offline.pth


100%|██████████| 97.2M/97.2M [00:02<00:00, 49.9MB/s]


In [ ]:
# ----------------- Define Queries -----------------
# Format: [frame_number, x, y]
# x, y are pixel coordinates in the ORIGINAL video resolution;
# they are rescaled automatically if the frames were downscaled.
# Example: track one point at frame 70, pixel (125, 50)

QUERY_FRAME = 70   # frame on which the object is selected

queries = torch.tensor([
    [QUERY_FRAME, 125., 50.],   # first point
    # [QUERY_FRAME, 650., 300.],   # second point
    # [QUERY_FRAME, 500., 270.],
], dtype=torch.float32, device=device)  # Send queries tensor to the same device (CPU or GPU)

if not (0 <= QUERY_FRAME < len(frames)):
    raise ValueError(f"QUERY_FRAME={QUERY_FRAME} is outside the video (0..{len(frames) - 1})")

# Rescale (x, y) to the size of the loaded frames
queries[:, 1:] *= scale

In [7]:
# ----------------- Run Model -----------------
# Run CoTracker model on the video with the defined query points.

#   pred_tracks     → predicted (x,y) positions of each query point across frames
#   pred_visibility → confidence (visibility score) for each tracked point
pred_tracks, pred_visibility = cotracker(video, queries=queries[None])

#================= Reshape outputs ===================#
#pred_tracks shape:  (B, T, N, 2) ------> (T, N, 2)
#   B = batch size (1 video here)
#   T = number of frames
#   N = number of query points
#   2 = (x, y) coordinates
# Remove batch dimension (B) → shape becomes (T, N, 2)
pred_tracks = pred_tracks.squeeze(0).cpu().numpy()

# pred_visibility shape: (B, T, N)
# Remove batch dimension (B) → shape becomes (T, N)
pred_visibility = pred_visibility.squeeze(0).cpu().numpy()

In [8]:
# ----------------- Debugging: Check data types and contents -----------------
print("frames:", type(frames))              # Should be <class 'numpy.ndarray'> after conversion
print("pred_tracks:", type(pred_tracks))    # Should be <class 'numpy.ndarray'> (tracking results)
print("pred_visibility:", type(pred_visibility))  # Should be <class 'numpy.ndarray'> (visibility scores)
print("queries:", type(queries))            # Should be <class 'torch.Tensor'> (queries defined on device)
print("queries content:", queries)          # Prints actual query tensor values (frame, x, y)

frames: <class 'numpy.ndarray'>
pred_tracks: <class 'numpy.ndarray'>
pred_visibility: <class 'numpy.ndarray'>
queries: <class 'torch.Tensor'>
queries content: tensor([[ 70., 125.,  50.]])


In [ ]:
#To check points you defined in queries really land in the right spots on your video

def show_initial_points(frame, queries, step=50, frame_idx=0):
    plt.figure(figsize=(10,6))       # Set figure size
    plt.imshow(frame)               # Display the frame (as an image)

    # ----------------- Extract query coordinates -----------------
    # If queries are a torch.Tensor → convert to NumPy
    if isinstance(queries, torch.Tensor):
        points = queries[:, 1:].cpu().numpy()  # take only (x, y), skip frame index
    else:
        points = np.array([q[1:] for q in queries])  # same for list/array input

    # ----------------- Plot query points -----------------
    plt.scatter(points[:,0], points[:,1], c="cyan", s=20, marker="o")  # plot circles

    # ----------------- Add labels near each point -----------------
    for i, (x,y) in enumerate(points):
        plt.text(x+5, y-5, str(i), color="cyan", fontsize=7, weight="bold")

    # ----------------- Set axis ticks -----------------
    # Tick marks every "step" pixels (to help locate points)
    H, W, _ = frame.shape
    plt.xticks(np.arange(0, W+1, step))
    plt.yticks(np.arange(0, H+1, step))

    # ----------------- Display settings -----------------
    plt.title(f"Initial Tracked Keypoints (Frame {frame_idx})")  # title of the plot
    plt.grid()
    plt.show()                                        # render the plot

# Visualize the query points on the frame where they were selected
show_initial_points(frames[QUERY_FRAME], queries, step=50, frame_idx=QUERY_FRAME)

In [ ]:
#========================== Main ===================#
VIS_THRESHOLD = 0.5   # a point is treated as visible when its visibility score is above this


def visible_runs(mask):
    """Return (start, end) index pairs (end exclusive) of consecutive True values."""
    runs, start = [], None
    for t, v in enumerate(mask):
        if v and start is None:
            start = t
        elif not v and start is not None:
            runs.append((start, t))
            start = None
    if start is not None:
        runs.append((start, len(mask)))
    return runs


def smooth_tracks(pred_tracks, pred_visibility, window=9, polyorder=3, vis_threshold=VIS_THRESHOLD):
    """
    Savitzky-Golay smoothing applied separately to each visible stretch of a track,
    so predictions made while the object is occluded do not leak into the visible path.
    """
    smoothed = pred_tracks.copy()
    T, N, _ = pred_tracks.shape
    for pid in range(N):
        for s, e in visible_runs(pred_visibility[:, pid] > vis_threshold):
            length = e - s
            win = min(window, length if length % 2 == 1 else length - 1)   # odd, <= run length
            if win > polyorder:
                smoothed[s:e, pid, :] = savgol_filter(pred_tracks[s:e, pid, :], win, polyorder, axis=0)
    return smoothed


def create_tracked_video(frames, pred_tracks, pred_visibility,
                         output_path="tracked_points.mp4", fps=30.0,
                         point_radius=4, line_thickness=2, smooth=True, seed=0):
    """Draw tracked points and their trajectories on the frames and save an .mp4.
    Returns the (smoothed) tracks that were drawn."""

    # ----------------- Setup -----------------
    num_frames, H, W, _ = frames.shape           # video dimensions
    num_points = pred_tracks.shape[1]            # number of query points

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")                  # codec for .mp4 files
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))   # same FPS as the source video
    if not out.isOpened():
        raise IOError(f"❌ Could not open video writer for: {output_path}")

    # One bright, distinct colour per tracked point (BGR), reproducible with a fixed seed
    rng = np.random.default_rng(seed)
    hues = (rng.permutation(num_points) * (180 // max(num_points, 1)) + rng.integers(0, 180)) % 180
    colors = [tuple(int(c) for c in cv2.cvtColor(np.uint8([[[h, 255, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])
              for h in hues]

    # ----------------- Smooth trajectories -----------------
    tracks = smooth_tracks(pred_tracks, pred_visibility) if smooth else pred_tracks.copy()

    # Each point's trajectory is a list of segments; a new segment starts after an
    # occlusion so no line is drawn across the gap.
    segments = [[] for _ in range(num_points)]
    was_visible = [False] * num_points

    # ----------------- Frame-by-frame drawing -----------------
    for t in range(num_frames):
        frame_bgr = cv2.cvtColor(frames[t], cv2.COLOR_RGB2BGR)     # OpenCV uses BGR

        for pid in range(num_points):
            visible = pred_visibility[t, pid] > VIS_THRESHOLD
            if visible:
                x, y = (int(round(v)) for v in tracks[t, pid])
                if not was_visible[pid]:
                    segments[pid].append([])            # object (re)appeared -> new segment
                segments[pid][-1].append((x, y))
            was_visible[pid] = visible

            # draw the full trajectory so far (all segments)
            pts = [np.array(seg, dtype=np.int32).reshape(-1, 1, 2) for seg in segments[pid] if len(seg) > 1]
            if pts:
                cv2.polylines(frame_bgr, pts, False, colors[pid], line_thickness, cv2.LINE_AA)

            # draw current position only while the point is visible
            if visible:
                cv2.circle(frame_bgr, (x, y), point_radius, colors[pid], -1, cv2.LINE_AA)

        # Write modified frame to output video
        out.write(frame_bgr)

    # ----------------- Finalize -----------------
    out.release()
    print(f"🎥 Tracked video saved as {output_path} ({fps:.2f} FPS)")
    return tracks

#calling function
smoothed_tracks = create_tracked_video(frames, pred_tracks, pred_visibility, OUTPUT_PATH, fps=fps)

In [ ]:
def evaluate_tracking(pred_tracks, pred_visibility, gt_tracks=None, gt_visibility=None, name=""):
    """
    Evaluates the performance of a tracking algorithm.

    Args:
        pred_tracks (np.ndarray): Predicted tracks of shape (T, N, D), where T is time, N is the number of tracks, and D is the dimensionality.
        pred_visibility (np.ndarray): Predicted visibility of shape (T, N), with values in [0, 1].
        gt_tracks (np.ndarray, optional): Ground truth tracks. Defaults to None.
        gt_visibility (np.ndarray, optional): Ground truth visibility. Defaults to None.
        name (str, optional): Label printed with the results (e.g. the video name).

    Note:
        Without ground truth only Smoothness / Continuity / Trajectory Length are computed.
        These describe how the track behaves, not how accurate it is.
    
    Returns:
        dict: A dictionary containing the evaluation metrics.
    """
    results = {}

    # --- With GT ---
    # This block executes only when ground truth data is provided for evaluation.
    if gt_tracks is not None:
        # Calculate the Mean Squared Error (MSE) between predicted and ground truth tracks.
        # MSE measures the average squared difference between the predictions and the ground truth.
        mse = np.mean((pred_tracks - gt_tracks)**2)

        # Calculate the End Point Error (EPE), which is the average Euclidean distance
        # between the predicted and ground truth track points.
        epe = np.linalg.norm(pred_tracks - gt_tracks, axis=-1).mean()

        # Define the threshold for the Percentage of Correct Keypoints (PCK) metric.
        threshold = 5

        # Calculate the Euclidean distance for each point to compare against the threshold.
        dist = np.linalg.norm(pred_tracks - gt_tracks, axis=-1)

        # Calculate PCK, which is the percentage of predicted points that are within
        # a certain distance (the threshold) of their corresponding ground truth points.
        pck = np.mean(dist < threshold)

        # Update the results dictionary with the calculated metrics.
        results.update({
            "MSE": mse,
            "EPE": epe,
            f"PCK@{threshold}px": pck,
        })

        # This inner block evaluates visibility accuracy if ground truth visibility is available.
        if gt_visibility is not None:
            # Binarize the predicted visibility using a threshold (0.5).
            pred_vis = (pred_visibility > 0.5).astype(int)

            # Calculate the accuracy of visibility prediction by comparing the binarized
            # predictions with the ground truth visibility.
            acc = (pred_vis == gt_visibility).mean()

            # Add visibility accuracy to the results.
            results["Visibility Accuracy"] = acc

    # --- Without GT ---
    # This block evaluates metrics that don't require ground truth data.
    # These metrics assess the internal quality of the predicted tracks.

    # Calculate smoothness. `np.diff(..., n=2)` computes the second-order difference,
    # which is the acceleration of the track. A smaller value indicates a smoother track.
    diffs = np.diff(pred_tracks, n=2, axis=0)
    smoothness = np.mean(np.linalg.norm(diffs, axis=-1))

    # Calculate continuity: fraction of (frame, point) pairs predicted visible (score > 0.5).
    # It represents the proportion of frames where a track is considered visible.
    continuity = (pred_visibility > 0.5).mean()
    
    # Calculate the average trajectory length. This is the sum of the Euclidean distances
    # between consecutive points for each track, averaged over all tracks.
    traj_length = np.mean([
        np.sum(np.linalg.norm(np.diff(pred_tracks[:, i, :], axis=0), axis=-1))
        for i in range(pred_tracks.shape[1])
    ])

    # Update the results dictionary with the no-GT metrics.
    results.update({
        "Smoothness": smoothness,
        "Continuity": continuity,
        "Trajectory Length": traj_length,
    })

    # Print a formatted summary of the evaluation results.
    print(f"📊 Evaluation Results: {name}" if name else "📊 Evaluation Results")
    for k, v in results.items():
        print(f"{k:20s}: {v:.4f}")

    # Return the dictionary of results.
    return results

In [ ]:
# Evaluate the tracks that were actually drawn (smoothed) ...
matrix = evaluate_tracking(smoothed_tracks, pred_visibility, name=f"{VIDEO_NAME} (smoothed)")
# ... and the raw model output for comparison
matrix_raw = evaluate_tracking(pred_tracks, pred_visibility, name=f"{VIDEO_NAME} (raw)")